
# 쿼리 최적화

논리적 최적화와 조건자 푸시다운을 사용하거나 사용하지 않는 예제를 포함하여 여러 예제에 대한 쿼리 계획 및 최적화를 살펴보겠습니다.

##### 목표
1. 논리적 최적화
1. 조건자 푸시다운
1. 조건자 푸시다운 사용 안 함

##### 방법
- <a href="https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.DataFrame.explain.html#pyspark.sql.DataFrame.explain" target="_blank">DataFrame</a>: **`explain`**


설정된 셀을 실행하고 초기 DataFrame을 **`df`** 변수에 저장해 보겠습니다. 이 DataFrame을 표시하면 이벤트 데이터가 표시됩니다.

In [0]:
%run ../Includes/Classroom-Setup-00.13

In [0]:
df = spark.read.table("events")
display(df)


### 논리적 최적화

**`explain(..)`**은 쿼리 계획을 출력하며, 선택적으로 지정된 설명 모드에 따라 형식이 지정됩니다. 다음 논리적 계획과 물리적 계획을 비교하면서 Catalyst가 여러 **`filter`** 변환을 어떻게 처리했는지 확인하세요.

In [0]:
from pyspark.sql.functions import col

limit_events_df = (df
                   .filter(col("event_name") != "reviews")
                   .filter(col("event_name") != "checkout")
                   .filter(col("event_name") != "register")
                   .filter(col("event_name") != "email_coupon")
                   .filter(col("event_name") != "cc_info")
                   .filter(col("event_name") != "delivery")
                   .filter(col("event_name") != "shipping_info")
                   .filter(col("event_name") != "press")
                  )

limit_events_df.explain(True)

물론, 원래는 단일 **`filter`** 조건을 사용하여 쿼리를 직접 작성할 수도 있었습니다. 이전 쿼리 계획과 다음 쿼리 계획을 비교해 보세요.

In [0]:
better_df = (df
             .filter((col("event_name").isNotNull()) &
                     (col("event_name") != "reviews") &
                     (col("event_name") != "checkout") &
                     (col("event_name") != "register") &
                     (col("event_name") != "email_coupon") &
                     (col("event_name") != "cc_info") &
                     (col("event_name") != "delivery") &
                     (col("event_name") != "shipping_info") &
                     (col("event_name") != "press"))
            )

better_df.explain(True)

물론, 의도적으로 다음 코드를 작성하지는 않겠지만, 길고 복잡한 쿼리에서는 중복된 필터 조건을 알아차리지 못할 수도 있습니다. Catalyst가 이 쿼리에서 어떤 작업을 수행하는지 살펴보겠습니다.

In [0]:
stupid_df = (df
             .filter(col("event_name") != "finalize")
             .filter(col("event_name") != "finalize")
             .filter(col("event_name") != "finalize")
             .filter(col("event_name") != "finalize")
             .filter(col("event_name") != "finalize")
            )

stupid_df.explain(True)


### Clean up classroom

In [0]:
DA.cleanup()